# 04 · Vectorial Search

Probamos las técnicas de recuperación vistas en clase:
1. **Similarity Search**: Top-K por similitud coseno (con score).
2. **Umbral mínimo** (score threshold): descarta resultados de baja calidad.
3. **MMR** (Maximal Marginal Relevance): diversidad en los resultados.
4. **Contexto formateado** para el prompt del LLM.

In [1]:
import sys
from pathlib import Path

# Asegura que el notebook encuentre los módulos del proyecto (config, rag_service, ...)
sys.path.insert(0, str(Path.cwd().parent))


In [2]:
from rag_service import RAGService

rag = RAGService()
if not rag.load_index():
    print("Primero ejecuta el notebook 03 para indexar los catálogos.")

Inicializando servicio RAG...
  Proveedor de embeddings: OPEN AI API
  Embeddings OPEN AI: text-embedding-3-small (dim=1536, batch=10)
Servicio RAG inicializado

Cargando índice desde: c:\Users\rmend\Dropbox\Personal_Desktop2026\BootCamp ML\DMC\Diplomado_AI_Engineering\DesignImplementation_Chatbots\Chatbot-AIGenerative-BankingSystem\data\faiss_index
Índice cargado exitosamente


In [4]:
# 1) SIMILARITY SEARCH: Top-K con score
query = "Cual es el fraude financiero más común en Perú en el sistema bancario?"
resultados = rag.search(query, k=5)

print(f"Query: {query}\n")
for i, r in enumerate(resultados, 1):
    print(f"{i}. [Score: {r['score']:.4f}] Fuente: {r['metadata'].get('source', 'N/A')}")
    print(f"   {r['text'][:180].replace(chr(10), ' ')}...")
    print()

Query: Cual es el fraude financiero más común en Perú en el sistema bancario?

1. [Score: 0.5456] Fuente: doc_202607141039292377.pdf
   tercero  sancionar al banco de crédito del perú con una multa de 11 60 uit27  once punto sesenta unidades impositivas tributarias  por infracción del artículo 19  de la l ey 29571 ...

2. [Score: 0.5387] Fuente: doc_202605211029049867.pdf
   i  que  el banco internacional del perú sociedad anónima abierta   interbank  el 22 de mayo de 2025  no habría adoptado las medidas de seguridad para impedir que se realice un cons...

3. [Score: 0.5368] Fuente: doc_202607011617092866.pdf
   i  por presunta infracción a lo establecido en el artículo 19 del código de protección y defensa del consumidor  en tanto  banco bbva perú s a  no habría cumplido con adoptar las m...

4. [Score: 0.5356] Fuente: doc_202607131311207079.pdf
   i  presunta infracción de los artículos 18  y 19  de la ley n  29571  código de protección y defensa del consumidor  en tanto  banco de cr

In [6]:
# 3) MMR: resultados relevantes pero diversos
query2 = "Cual es el fraude financiero más común en Perú en el sistema bancario?"
print(f"Query: {query2}\n")

print("--- Similarity Search (puede repetir la misma fuente) ---")
for i, r in enumerate(rag.search(query2, k=5, threshold=0.0), 1):
    print(f"  {i}. {r['metadata'].get('source', '?')} | {r['text'][:80].replace(chr(10), ' ')}...")

print("\n--- MMR (lambda_mult=0.7: relevancia + diversidad) ---")
for i, r in enumerate(rag.search_mmr(query2, k=5, lambda_mult=0.7), 1):
    print(f"  {i}. {r['metadata'].get('source', '?')} | {r['text'][:80].replace(chr(10), ' ')}...")

Query: Cual es el fraude financiero más común en Perú en el sistema bancario?

--- Similarity Search (puede repetir la misma fuente) ---
  1. doc_202607141039292377.pdf | tercero  sancionar al banco de crédito del perú con una multa de 11 60 uit27  on...
  2. doc_202605211029049867.pdf | i  que  el banco internacional del perú sociedad anónima abierta   interbank  el...
  3. doc_202607011617092866.pdf | i  por presunta infracción a lo establecido en el artículo 19 del código de prot...
  4. doc_202607131311207079.pdf | i  presunta infracción de los artículos 18  y 19  de la ley n  29571  código de ...
  5. doc_202603061200515478.pdf | primero  admitir a trámite la denuncia del 5 de agosto de 2025  interpuesta por ...

--- MMR (lambda_mult=0.7: relevancia + diversidad) ---
  1. doc_202607141039292377.pdf | tercero  sancionar al banco de crédito del perú con una multa de 11 60 uit27  on...
  2. doc_202607141039292377.pdf | sobre el patrón de fraude...
  3. doc_202607011617092866.pdf | tr

In [8]:
# 4) CONTEXTO PARA EL LLM: formato listo para el prompt RAG
contexto = rag.get_context_for_query("¿Cual es el fraude financiero mas sancionado a Banco de Crédito de ´Perú?", 
                                     k=5)
print(contexto)
print("...")

[Documento 1 - Fuente: doc_202607141039292377.pdf - Relevancia: 0.64]
tercero  sancionar al banco de crédito del perú con una multa de 11 60 uit27  once
punto sesenta unidades impositivas tributarias  por infracción del artículo 19  de la l ey
29571  código de protección y defensa del consumidor  por no adoptar medidas de
seguridad respecto de una operación de transferencia realizada el 03 de octubre de 2025 con
cargo a la cuenta de ahorros n  285    0 29 de la denunciante por el monto de s 
23 150 00

---

[Documento 2 - Fuente: doc_202603041208115643.pdf - Relevancia: 0.62]
cuarto  sancionar a banco de crédito del perú s a  conforme a lo siguiente 

cuadro n 1  detalle de sanciones

---

[Documento 3 - Fuente: doc_202603041208115643.pdf - Relevancia: 0.61]
sanción   banco de crédito del perú s a  once con sesenta
 11 60  unidades impositivas tributarias  uit 
banco de crédito del perú s a  cero con
cincuenta y dos  0 52  unidades impositivas
tributarias  uit 

lima  25 de febrero de 